In [5]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

Check how many documents are in the index:

In [6]:
sqlite_index.count()

153

Try to search a document:

In [8]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Can I submit homework after the deadline, or get a deadline extension?',
 'I missed the first homework - can I still get a certificate?']

### RAG with sqlitesearch

We use the RAGBase class from `rag_helper.py` with this sqlitesearch index.

Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

In [10]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, so we just connect to the database file.

In [11]:
openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [12]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it has started. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


The answer should be similar to what we got with minsearch. But now the data comes from a persistent index - no fetching, no processing, no indexing at startup. And we didn't have to rewrite any of the RAG logic - just swapped the index.

The modular design splits the work cleanly:

- `ingest.py` handles data loading and indexing
- `rag_helper.py` handles the RAG pipeline
- the notebooks wire them together

With minsearch (single process):

```
Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
```

With sqlitesearch (two processes):

```
Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready
```

The full architecture:

```mermaid
flowchart TD

    subgraph ING["INGESTION"]
        direction LR
        FAQ[FAQ.json]
        INGESTOR[Ingestor<br/>parse, chunk, embed, metadata]
        FAQ --> INGESTOR
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    INGESTOR -->|Index Documents| DB
```

The RAG assistant then reads from it:

```mermaid
flowchart TD

    subgraph RAG["RAG ASSISTANT"]
        U([🙂 User])
        APP[Application]
        DOCS[[D1 ... D5]]
        PROMPT[Build Prompt<br/>Question + Context]
        LLM[LLM]
        ANSWER([Answer])

        U -->|Question| APP
        DOCS --> APP
        APP --> PROMPT
        PROMPT --> LLM
        LLM --> ANSWER
        ANSWER --> U
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
```

For larger production systems, use the same pattern with a different backend:

- Elasticsearch
- OpenSearch
- Qdrant (vector database)
- Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

When you're done, close the database connection:

In [13]:
sqlite_index.close()

Or just let Python clean it up when the notebook kernel shuts down.